# Feature Engineering

Use this notebook to derive model-ready features and document transformation logic.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

import pickle
import json

In [2]:
df = pd.read_csv("../data/processed/Bacteria_dataset_cleaned.csv")

In [3]:
print(df.shape)
df.dtypes

(10710, 22)


Diabetes             float64
Hypertension         float64
Hospital_before      float64
Infection_Freq       float64
AMX/AMP              float64
AMC                  float64
CZ                   float64
FOX                  float64
CTX/CRO              float64
IPM                  float64
GEN                  float64
AN                   float64
Acide nalidixique    float64
ofx                  float64
CIP                  float64
C                    float64
Co-trimoxazole       float64
Furanes              float64
colistine            float64
Age                  float64
Gender               float64
Bacteria              object
dtype: object

## Analysing missing data

In [4]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})

missing_summary

,Missing Count,Missing %
Diabetes,763,7.12
Hypertension,763,7.12
Hospital_before,763,7.12
Infection_Freq,996,9.30
AMX/AMP,753,7.03
AMC,753,7.03
CZ,753,7.03
FOX,753,7.03
CTX/CRO,753,7.03
IPM,753,7.03


In [5]:
numeric_features = [
    "Age",
    "Infection_Freq"
]

binary_features = [
    "Gender",
    "Diabetes",
    "Hypertension",
    "Hospital_before"
]

categorical_features = [
    "Bacteria"
]

required_features = [
    "Age",
    "Gender",
    "Diabetes",
    "Hypertension",
    "Hospital_before",
    "Bacteria"
]

## Removing colmns that are entirely empty

In [6]:
rows_before = len(df)
df = df.dropna(subset=required_features)

rows_after = len(df)

print(f"Rows before : {rows_before}")
print(f"Rows after  : {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

Rows before : 10710
Rows after  : 9947
Rows removed: 763


In [7]:
print(df[required_features].isna().sum())

print("\nRemaining missing values:")

print(df[numeric_features].isna().sum())

Age                0
Gender             0
Diabetes           0
Hypertension       0
Hospital_before    0
Bacteria           0
dtype: int64

Remaining missing values:
Age                 0
Infection_Freq    233
dtype: int64


## Separtaing feature and target colmn

In [8]:
feature_columns = [
    "Age",
    "Gender",
    "Diabetes",
    "Hypertension",
    "Hospital_before",
    "Infection_Freq",
    "Bacteria"
]

In [9]:
target_columns = [
    "AMX/AMP",
    "AMC",
    "CZ",
    "FOX",
    "CTX/CRO",
    "IPM",
    "GEN",
    "AN",
    "Acide nalidixique",
    "ofx",
    "CIP",
    "C",
    "Co-trimoxazole",
    "Furanes",
    "colistine"
]

In [10]:
X = df[feature_columns].copy()
Y = df[target_columns].copy()

print(X.shape,Y.shape)
print(X.columns.tolist())
print(Y.columns.tolist())

(9947, 7) (9947, 15)
['Age', 'Gender', 'Diabetes', 'Hypertension', 'Hospital_before', 'Infection_Freq', 'Bacteria']
['AMX/AMP', 'AMC', 'CZ', 'FOX', 'CTX/CRO', 'IPM', 'GEN', 'AN', 'Acide nalidixique', 'ofx', 'CIP', 'C', 'Co-trimoxazole', 'Furanes', 'colistine']


## 0  Susceptible &  1	Non-Susceptible (Intermediate or Resistant)

In [11]:
Y = Y.replace({
    2: 1
})

# Train Test Split

70% training data, 
15% validation set for tuning, 
15% test set

In [12]:
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X,
    Y,
    test_size=0.30,
    random_state=42,
    shuffle=True,
    stratify=X["Bacteria"]   # Preserve bacterial species distribution across splits
)

In [13]:
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.50,
    random_state=42,
    shuffle=True,
    stratify=X_temp["Bacteria"]   # Maintain the same bacterial distribution in validation and test sets
)

In [14]:
bacteria_encoder = LabelEncoder()

In [15]:
bacteria_encoder.fit(X_train["Bacteria"])

LabelEncoder()

In [16]:
X_train["Bacteria"] = bacteria_encoder.transform(X_train["Bacteria"])
X_val["Bacteria"] = bacteria_encoder.transform(X_val["Bacteria"])
X_test["Bacteria"] = bacteria_encoder.transform(X_test["Bacteria"])

In [17]:
mapping = pd.DataFrame({
    "Encoded": range(len(bacteria_encoder.classes_)),
    "Bacteria": bacteria_encoder.classes_
})

mapping

,Encoded,Bacteria
0,0,Acinetobacter baumannii
1,1,Citrobacter spp.
2,2,Enterobacteria spp.
3,3,Escherichia coli
4,4,Klebsiella pneumoniae
5,5,Morganella morganii
6,6,Proteus mirabilis
7,7,Pseudomonas aeruginosa
8,8,Serratia marcescens


### Impute the na values of infection freq with median

In [18]:
from sklearn.impute import SimpleImputer
infection_imputer = SimpleImputer(strategy="median")

In [19]:
infection_imputer.fit(
    X_train[["Infection_Freq"]]
)

,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [20]:
X_train["Infection_Freq"] = infection_imputer.transform(X_train[["Infection_Freq"]])
X_val["Infection_Freq"] = infection_imputer.transform(X_val[["Infection_Freq"]])
X_test["Infection_Freq"] = infection_imputer.transform(X_test[["Infection_Freq"]])

### Scaling the numerical values

In [21]:
clinical_scaler = StandardScaler()
clinical_scaler.fit(X_train[numeric_features])

,copy,True
,with_mean,True
,with_std,True


In [23]:
X_train[numeric_features] = clinical_scaler.transform(X_train[numeric_features])
X_val[numeric_features] = clinical_scaler.transform(X_val[numeric_features])
X_test[numeric_features] = clinical_scaler.transform(X_test[numeric_features])

In [24]:
print(X_train[numeric_features].agg(["mean", "std"])) #for validation

               Age  Infection_Freq
mean -8.675112e-17   -8.522022e-17
std   1.000072e+00    1.000072e+00


### final validation

In [25]:
print("Training")
print("X:", X_train.shape)
print("Y:", Y_train.shape)

print("Validation")
print("X:", X_val.shape)
print("Y:", Y_val.shape)

print("Testing")
print("X:", X_test.shape)
print("Y:", Y_test.shape)

Training
X: (6962, 7)
Y: (6962, 15)
Validation
X: (1492, 7)
Y: (1492, 15)
Testing
X: (1493, 7)
Y: (1493, 15)


In [26]:
print(X_train.dtypes)

Age                float64
Gender             float64
Diabetes           float64
Hypertension       float64
Hospital_before    float64
Infection_Freq     float64
Bacteria             int64
dtype: object


In [27]:
print("Training Missing Values")
print(X_train.isna().sum())

print("\nValidation Missing Values")
print(X_val.isna().sum())

print("\nTesting Missing Values")
print(X_test.isna().sum())

Training Missing Values
Age                0
Gender             0
Diabetes           0
Hypertension       0
Hospital_before    0
Infection_Freq     0
Bacteria           0
dtype: int64

Validation Missing Values
Age                0
Gender             0
Diabetes           0
Hypertension       0
Hospital_before    0
Infection_Freq     0
Bacteria           0
dtype: int64

Testing Missing Values
Age                0
Gender             0
Diabetes           0
Hypertension       0
Hospital_before    0
Infection_Freq     0
Bacteria           0
dtype: int64


## saving the processed dataset

In [30]:
from pathlib import Path
import numpy as np
import pickle
import json

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

np.save(artifact_dir / "X_train.npy", X_train.to_numpy())
np.save(artifact_dir / "X_val.npy", X_val.to_numpy())
np.save(artifact_dir / "X_test.npy", X_test.to_numpy())

np.save(artifact_dir / "Y_train.npy", Y_train.to_numpy())
np.save(artifact_dir / "Y_val.npy", Y_val.to_numpy())
np.save(artifact_dir / "Y_test.npy", Y_test.to_numpy())


# Saved Processed DataFrames (Useful for Debugging)
X_train.to_csv(artifact_dir / "X_train.csv", index=False)
X_val.to_csv(artifact_dir / "X_val.csv", index=False)
X_test.to_csv(artifact_dir / "X_test.csv", index=False)

Y_train.to_csv(artifact_dir / "Y_train.csv", index=False)
Y_val.to_csv(artifact_dir / "Y_val.csv", index=False)
Y_test.to_csv(artifact_dir / "Y_test.csv", index=False)

with open(artifact_dir / "clinical_scaler.pkl", "wb") as file:
    pickle.dump(clinical_scaler, file)

with open(artifact_dir / "infection_imputer.pkl", "wb") as file:
    pickle.dump(infection_imputer, file)

with open(artifact_dir / "bacteria_encoder.pkl", "wb") as file:
    pickle.dump(bacteria_encoder, file)

with open(artifact_dir / "feature_columns.json", "w") as file:
    json.dump(feature_columns, file, indent=4)

with open(artifact_dir / "target_columns.json", "w") as file:
    json.dump(target_columns, file, indent=4)

bacteria_mapping = {
    int(index): bacteria
    for index, bacteria in enumerate(bacteria_encoder.classes_)
}

with open(artifact_dir / "bacteria_mapping.json", "w") as file:
    json.dump(bacteria_mapping, file, indent=4)

dataset_metadata = {
    "train_samples": len(X_train),
    "validation_samples": len(X_val),
    "test_samples": len(X_test),
    "num_features": X_train.shape[1],
    "num_targets": Y_train.shape[1],
    "numeric_features": numeric_features,
    "binary_features": binary_features,
    "categorical_features": categorical_features,
    "feature_columns": feature_columns,
    "target_columns": target_columns,
    "bacteria_classes": len(bacteria_encoder.classes_),
    "random_seed": 42,
    "train_split": 0.70,
    "validation_split": 0.15,
    "test_split": 0.15,
    "scaler": "StandardScaler",
    "imputer": "Median (Infection_Freq)",
    "target_encoding": {
        "0": "Susceptible",
        "1": "Intermediate/Resistant"
    }
}

with open(artifact_dir / "dataset_metadata.json", "w") as file:
    json.dump(dataset_metadata, file, indent=4)

print("saved successfully")
print(f"Location: {artifact_dir.resolve()}")

saved successfully
Location: C:\Users\Hridesh\Sentinel-GNN\backend\notebooks\artifacts
